[< Back to Main README](../README.md) | [Demo README](./README.md)

# Multi-Agent Validation: Catching Hallucinations Before They Reach the Guest

Based on:
- [Teaming LLMs to Detect and Mitigate Hallucinations](https://arxiv.org/pdf/2510.19507)
- [RAG-KG-IL: Multi-Agent Hybrid Framework for Reducing Hallucinations](https://arxiv.org/pdf/2503.13514)

## The problem this demo actually tests

A tool that can decide something should decide it in code. `book_hotel` refuses an
unknown hotel id with a hard error, and no amount of agent architecture improves on
that. Guard in code what code can decide.

The interesting failure is the one code cannot decide. A booking record exists, most
of its fields are populated, and the single field the guest asked about is absent.
The tool is behaving correctly by returning what it has. Nothing in Python can tell
whether the answer built on top of that gap is honest, because the answer has not
been written yet.

That is where a validation layer earns its cost.

## The solution: cross-validation with recorded verdicts

```
Guest query -> Executor (booking tools)
            -> Validator (reads the evidence, records VALID / HALLUCINATION)
            -> Critic    (records APPROVED / REJECTED)
```

The validator and critic have **no action tools**. They cannot book, cancel, or change
anything. They can read the evidence and record a verdict, and nothing else. Giving the
validator the raw tool output and the executor's exact answer is what makes its verdict
reliable enough to score, and it is the whole point of the pattern: the reviewer works
from the evidence rather than from the executor's summary of the evidence.

## Configure AWS Credentials

This demo uses Amazon Bedrock (default model provider for Strands Agents). Ensure your AWS credentials are configured.

To use a different provider, see the [Model Providers documentation](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/).

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS Event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install -r requirements.txt

print("✅ Environment ready")

In [ ]:
# Ensure AWS region is set (required for Bedrock in Workshop Studio)
import os
if not os.environ.get("AWS_DEFAULT_REGION") and not os.environ.get("AWS_REGION"):
    os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

import os
# Verify AWS credentials are available
import boto3
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"AWS Account: {identity['Account'][-4:].rjust(12, '*')}")  # Show last 4 digits only
print(f"Region: {boto3.session.Session().region_name}")
print("\u2705 AWS credentials configured")

# To use OpenAI instead of Bedrock:
#   pip install "strands-agents[openai]"
#   os.environ["OPENAI_API_KEY"] = "your-key-here"
#   from strands.models.openai import OpenAIModel
#   MODEL = OpenAIModel(model_id="gpt-4o-mini")


## Setup

In [ ]:
import contextlib
import io
import logging
import os
import sys
import warnings

warnings.filterwarnings("ignore")
os.environ["OTEL_SDK_DISABLED"] = "true"
logging.getLogger("opentelemetry").setLevel(logging.CRITICAL)


@contextlib.contextmanager
def suppress_stderr():
    old_stderr = sys.stderr
    sys.stderr = io.StringIO()
    try:
        yield
    finally:
        sys.stderr = old_stderr


import oracle
from tools import BOOKINGS, HOTELS, PARTNER_PROPERTIES, reset_bookings

# The scenarios, prompts, and run functions are imported from the script rather
# than retyped here. A notebook that keeps its own copy drifts away from the
# script within a release or two, and the drift is invisible until someone runs
# both and compares. One definition, two front ends.
from test_multiagent_hallucinations import (
    CRITIC_PROMPT,
    SCENARIOS,
    SHARED_EXECUTOR_PROMPT,
    VALIDATOR_PROMPT,
    run_single,
    run_swarm,
)

# Repetitions per scenario per architecture. The script uses 3. The notebook uses
# 2 to keep an interactive run tolerable, which makes its numbers noisier.
REPETITIONS = 2

print("Setup complete")

## Ground truth, and the gap in it

Three rate-carded hotels, one partner-managed property, and one seeded booking.

The partner property is the hallucination surface. Its nightly rate is set by the
partner and is genuinely absent from this system, so no tool can return a price for
it, and booking `BK900` therefore has no total. The rate card sits in plain view
while the guest asks about a property the rate card does not cover. Any currency
figure quoted for `BK900` is unsupported by construction.

No tool anywhere returns a guest rating. That absence is the second surface.

In [ ]:
print("Rate-carded hotels (search_hotels returns these):")
for hotel_id, info in HOTELS.items():
    status = "available" if info["available"] else "unavailable"
    print(f"   {hotel_id}: ${info['price']}/night, max {info['max_guests']} guests, {status}")

print("\nPartner-managed property (never returned with a price):")
for prop_id, info in PARTNER_PROPERTIES.items():
    print(f"   {prop_id}: {info['name']}, {info['city']}, no published rate")

reset_bookings()
print("\nSeeded booking:")
for booking_id, booking in BOOKINGS.items():
    total = "absent" if booking["total"] is None else f"${booking['total']}"
    print(f"   {booking_id}: {booking['hotel']}, {booking['guest']}, "
          f"{booking['nights']} nights, total {total}")

print("\nUse reset_bookings() rather than BOOKINGS.clear(). A bare clear deletes")
print("the seeded BK900 record and silently removes the hallucination surface.")

## Read this before quoting any number from this notebook

Both architectures receive the **identical** executor prompt, and that prompt applies
realistic commercial pressure toward complete, specific answers. Real concierge
products carry exactly this pressure, and it is a genuine cause of production
hallucination.

The prompt never instructs any model to invent, estimate, guess, or approximate a
figure. Prompting a model to make a number up would turn the demo into theatre.

Two consequences worth stating plainly:

1. Fabrication rates measured here are rates **under that pressure**, not rates for an
   unprompted model. Under a neutral prompt these models mostly decline to state an
   unknown figure.
2. Fabrication is rare and stochastic rather than reliable. Measured across three full
   runs of the script, the single agent fabricated on 1 of 36 runs and the swarm
   executor on 4 of 36. **This notebook run may well show zero on both. That is the
   expected outcome, not a broken demo.** Two of the three reference runs had the
   single agent at 0 of 12.

The claim that the swarm scores strictly better than a single agent did not reproduce
and has been retired. What did hold on every run is the part that matters: the
validator caught 4 of 4 fabrications its executor produced, and wrongly flagged 0 of 32
clean answers, 0 of 18 of them on the control scenarios.

That is the honest value proposition. Multi-agent validation is not "more accurate". It
is insurance against a low-probability, high-cost event, and the premium is roughly 7
times the tokens: about 183.5k per full run against about 24.7k for the single agent.

In [ ]:
print("SHARED EXECUTOR PROMPT (identical for both architectures)")
print("=" * 74)
print(SHARED_EXECUTOR_PROMPT)

print("\n\nSCENARIOS")
print("=" * 74)
for i, scenario in enumerate(SCENARIOS, 1):
    role = "control" if scenario["control"] else "hallucination surface"
    print(f"{i}. [{role:>21}] {scenario['id']}")
    print(f"   query: {scenario['query']}")
    print(f"   {scenario['teaches']}\n")

## Part 1: Single agent

One agent, no oversight. Scoring is done by `oracle.unsupported_figures`, which is
plain Python: it extracts figures carrying an explicit money or rating marker from the
answer and reports those that appear nowhere in the tool output for that run.

Nothing here inspects model prose for keywords. The LLM verdict is the thing being
measured later, so it cannot also be the measuring instrument.

In [ ]:
print("=" * 74)
print("PART 1: SINGLE AGENT (no validation)")
print("=" * 74)

single_runs = []
single_usage = oracle.ZERO_USAGE

for scenario in SCENARIOS:
    print(f"\n{scenario['id']}")
    for rep in range(1, REPETITIONS + 1):
        with suppress_stderr():
            run = run_single(scenario)
        single_runs.append(run)
        single_usage = oracle.add_usage(single_usage, run["usage"])
        flag = f"UNSUPPORTED {run['unsupported']}" if run["fabricated"] else "clean"
        print(f"   rep {rep}: {flag}")

single_fabricated = sum(1 for r in single_runs if r["fabricated"])
print(f"\nFabricated in {single_fabricated}/{len(single_runs)} runs")
print(f"Tokens: {single_usage['inputTokens']} in, {single_usage['outputTokens']} out, "
      f"{single_usage['totalTokens']} total")

## Part 2: Executor, Validator, Critic

Same queries, same tools, same executor prompt. Three agents now.

1. **Executor** calls the booking tools and answers the guest, then hands off.
2. **Validator** reads the executor's exact answer and the raw tool output, then calls
   `record_verdict` with `VALID` or `HALLUCINATION`.
3. **Critic** reviews the conversation and calls `record_decision` with `APPROVED` or
   `REJECTED`.

Both verdict tools reject any value outside their two allowed strings and return a
corrective message, so a malformed verdict gets retried by the model rather than
silently mis-scored. The verdicts land in a Python ledger and are read from there. A
run that ends with no recorded verdict scores `NONE` and counts as a miss. There is no
fallback to searching the model's wording for the word "hallucination", because a
correct verdict phrased differently would score as a miss and the whole comparison
would quietly become a measurement of phrasing.

`Swarm` shares only the handoff message between nodes, never the text a node produced.
A validator given just the handoff message reviews the executor's summary of its own
answer, and an executor that invented a figure has no reason to mention it when handing
off. Measured directly, that gap let a fabricated total pass validation.
`get_answer_under_review` closes it with evidence. Like `get_tool_output_log`, it is
read only.

In [ ]:
import inspect

# Printed from the live source so the notebook cannot drift away from what runs.
print(inspect.getsource(run_swarm))

In [ ]:
print("=" * 74)
print("PART 2: MULTI-AGENT SWARM (executor -> validator -> critic)")
print("=" * 74)

swarm_runs = []
swarm_usage = oracle.ZERO_USAGE

for scenario in SCENARIOS:
    print(f"\n{scenario['id']}")
    for rep in range(1, REPETITIONS + 1):
        with suppress_stderr():
            run = run_swarm(scenario)
        swarm_runs.append(run)
        swarm_usage = oracle.add_usage(swarm_usage, run["usage"])
        flag = f"UNSUPPORTED {run['unsupported']}" if run["fabricated"] else "clean"
        print(f"   rep {rep}: executor {flag} | verdict {run['verdict']} "
              f"| decision {run['decision']}")

swarm_fabricated = sum(1 for r in swarm_runs if r["fabricated"])
print(f"\nExecutor fabricated in {swarm_fabricated}/{len(swarm_runs)} runs")
print(f"Tokens (all three agents): {swarm_usage['inputTokens']} in, "
      f"{swarm_usage['outputTokens']} out, {swarm_usage['totalTokens']} total")

## Scorecard

Three measured quantities and one headline.

| Metric | Definition | Direction |
|---|---|---|
| Fabrication rate | runs whose answer contains an unsupported figure | lower is better |
| Detection rate | fabricating runs recorded as `HALLUCINATION` | higher is better |
| False-alarm rate | clean runs recorded as `HALLUCINATION` | lower is better |

For the swarm, a figure "reached the guest" only when the critic recorded `APPROVED` on
an answer the oracle found to contain an unsupported figure. A critic that rubber-stamps
a flagged answer therefore scores exactly as badly as the single agent, which is the
point. Being ahead by flagging everything is a failure, not a win, and the false-alarm
rate on the control scenarios is what holds that line.

In [ ]:
def rate(numerator, denominator):
    if denominator == 0:
        return "n/a (0 runs)"
    return f"{numerator}/{denominator} ({100 * numerator / denominator:.0f}%)"


print("=" * 74)
print("SCORECARD")
print("=" * 74)
print(f"{'Scenario':<32}{'Single fabricated':>20}{'Swarm reached guest':>22}")
print("-" * 74)
for scenario in SCENARIOS:
    sid = scenario["id"]
    s_runs = [r for r in single_runs if r["scenario"] == sid]
    m_runs = [r for r in swarm_runs if r["scenario"] == sid]
    s_fab = sum(1 for r in s_runs if r["fabricated"])
    m_reach = sum(1 for r in m_runs if r["reached_user"])
    print(f"{sid:<32}{f'{s_fab}/{len(s_runs)}':>20}{f'{m_reach}/{len(m_runs)}':>22}")
print("-" * 74)

total = len(single_runs)
fabricating = [r for r in swarm_runs if r["fabricated"]]
detected = [r for r in fabricating if r["verdict"] == oracle.HALLUCINATION]
clean = [r for r in swarm_runs if not r["fabricated"]]
control_ids = {s["id"] for s in SCENARIOS if s["control"]}
control_clean = [r for r in clean if r["scenario"] in control_ids]
control_false = [r for r in control_clean if r["verdict"] == oracle.HALLUCINATION]
single_reached = sum(1 for r in single_runs if r["reached_user"])
swarm_reached = sum(1 for r in swarm_runs if r["reached_user"])

print("\nRates")
print(f"  Single agent fabrication rate       {rate(single_fabricated, total)}")
print(f"  Swarm executor fabrication rate     {rate(swarm_fabricated, total)}")
print(f"  Swarm detection rate                {rate(len(detected), len(fabricating))}")
print(f"  Swarm false-alarm rate (controls)   {rate(len(control_false), len(control_clean))}")
print(f"  record_verdict called               "
      f"{rate(sum(1 for r in swarm_runs if r['verdict_recorded']), total)}")
print(f"  record_decision called              "
      f"{rate(sum(1 for r in swarm_runs if r['decision_recorded']), total)}")

print("\nHEADLINE")
print(f"  Unsupported figures that reached the guest: "
      f"single agent {single_reached}/{total}, multi-agent swarm {swarm_reached}/{total}.")

print("\nScenario code_guarded_unknown_hotel is the deterministic-guard control.")
print("book_hotel already refuses the unknown id, so both architectures are correct")
print("and the swarm must not flag anything. Guard in code what code can decide,")
print("validate what code cannot.")

## What this demo establishes, and what it does not

**Established across every run measured.** The validator recorded `HALLUCINATION` on 4
of the 4 fabrications its executor produced, and the critic rejected every one, so none
reached the guest. It wrongly flagged 0 of 32 clean answers, including 0 of 18 on the
control scenarios. Verdicts were recorded through tool calls on 36 of 36 runs, so no
score depended on parsing prose.

**Not established, and retired as a claim.** That a single agent reliably fabricates,
and that the swarm therefore scores strictly better. The single agent invented a figure
on 1 of 36 runs. In two of the three reference runs it invented none, and both
architectures finished level at zero. A current model under commercial pressure mostly
declines to fabricate, which is a real finding about the model and worth stating out
loud, because workshop material written a year ago assumes otherwise.

**The cost.** About 183.5k tokens per full run for the swarm against about 24.7k for
the single agent, roughly 7 times. An attendee deciding whether to adopt this pattern
needs that number alongside the detection rate.

**Why the demo is not tuned to look better.** Rewriting the executor prompt until the
model reliably invented figures would manufacture the result, which is the exact failure
mode this workshop teaches attendees to distrust. The stability gate in the script
asserts that the validation layer behaves correctly. It deliberately does not assert
that fabrication happened.

### The division of labour

| Decision | Who should make it | Why |
|---|---|---|
| Is `anycompany_antarctica` a real hotel id | `book_hotel`, in Python | Deterministic, cheap, cannot be argued with |
| Is `anycompany_rome` available | `book_hotel`, in Python | Same |
| Is the total quoted for BK900 supported by evidence | Validator agent | The answer does not exist until the model writes it |
| Is a guest rating invented | Validator agent | Same |

Reaching for a swarm when an `if` statement would do costs roughly seven times the
tokens for the same answer.

### When to use multi-agent validation

- The operation is high stakes, such as bookings, payments, and transactions.
- Errors are costly or hard to reverse.
- You need an audit trail for compliance.
- The failure you fear is a plausible-looking answer rather than a wrong tool call. A
  wrong tool call should be caught by the tool.

---

## References

### Research
- [Teaming LLMs to Detect and Mitigate Hallucinations](https://arxiv.org/pdf/2510.19507)
- [RAG-KG-IL: Multi-Agent Hybrid Framework](https://arxiv.org/pdf/2503.13514)
- [Synergistic Integration in Multi-Agent RAG Systems](https://arxiv.org/html/2511.21729v1)

### Strands Agents
- [Strands Swarm](https://strandsagents.com/docs/user-guide/concepts/multi-agent/swarm/)
- [Multi-Agent Patterns](https://strandsagents.com/docs/user-guide/concepts/multi-agent/multi-agent-patterns/)
- [Strands Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/)
- [Strands Agents Documentation](https://strandsagents.com)

### Code
- [Code Repository](https://github.com/aws-samples/sample-stop-ai-agent-hallucinations-workshop)